In [ ]:
import os
import time
import urllib.request
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
import pandas as pd
import numpy as np
from pytesseract import Output
import pytesseract
import cv2
import io

from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup

import urllib.request

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import NoSuchElementException

from PIL import Image, ImageOps, ImageFilter
from skimage import io
from skimage.filters import threshold_otsu
from skimage.morphology import remove_small_objects

#텍스트 전처리
import re
import nltk
from nltk.corpus import stopwords

## 이미지 크롤링 후 OCR 진행

In [ ]:
def initialize_driver():
    chrome_options = webdriver.ChromeOptions()
    driver = webdriver.Chrome(service=Service("chromedriver"), options=chrome_options)
    return driver

def get_theater_links(driver, query):
    driver.get(query)
    time.sleep(1)
    number = 160

    addresslist = []

    for i in range(1, int(number) + 1):
        xpath = f"/html/body/table/tbody/tr[2]/td[3]/div/div/div[2]/div/table/tbody/tr[{i}]/td[1]/a"
        elem = driver.find_element(By.XPATH, xpath).get_attribute("href")
        addresslist.append(elem)

    return addresslist


def resize_image(image, max_height):
    # 이미지 크기를 최대 높이에 맞춰 조정
    width_percent = (max_height / float(image.size[1]))
    new_width = int((float(image.size[0]) * float(width_percent)))
    resized_image = image.resize((new_width, max_height), Image.LANCZOS)
    return resized_image

def image(driver, addresslist, save_directory, max_image_height=2000):
    scraped_image = []
    for index, lis in enumerate(addresslist):
        driver.get(lis)
        time.sleep(1)

        # 이미지 요소 찾기
        img_area_elements = driver.find_elements(By.XPATH, '//*[@id="productMainBody"]/div/div/div[5]/div/p/strong/img')
        title_element = driver.find_element(By.XPATH, '//*[@id="container"]/div[5]/div[1]/div[2]/div[1]/div/div[1]/h2')

        # 이미지가 없을 때 처리
        if not img_area_elements:
            img_area_elements = driver.find_elements(By.XPATH, '//*[@id="productMainBody"]/div/div/div[3]/div/p/strong/img')
            if not img_area_elements:
                img_area_elements = driver.find_elements(By.XPATH, '//*[@id="productMainBody"]/div/div/div[4]/div/p/strong/img')
            # 이미지 저장
            image_list = []
            for i, img_element in enumerate(img_area_elements):
                img_url = img_element.get_attribute("src")
                img_data = urllib.request.urlopen(img_url).read()

                # 파일 이름 생성
                filename=title_element.text.replace('/', '_').replace(':', '_')
                file_name= filename+"_part"+str(i + 1)+".png"
                file_path = os.path.join(save_directory, file_name)

                # 이미지 저장
                with open(file_path, "wb") as file:
                    file.write(img_data)
                    print(f"Saved {file_name}")

        else:
            # 이미지가 있는 경우 처리 코드

            # 이미지 저장
            image_list = []
            for i, img_element in enumerate(img_area_elements):
                img_url = img_element.get_attribute("src")
                img_data = urllib.request.urlopen(img_url).read()

                # 파일 이름 생성
                filename=title_element.text.replace('/', '_').replace(':', '_')
                file_name= filename+"_part"+str(i + 1)+".png"
                file_path = os.path.join(save_directory, file_name)

                # 이미지 저장
                with open(file_path, "wb") as file:
                    file.write(img_data)
                    print(f"Saved {file_name}")

        #7.이미지에서 문자추출
        for i, img_element in enumerate(img_area_elements):
            img_text=[]
            img_name='D:/캡스톤 디자인/연극포스터/'+filename+"_part"+str(i + 1)+".png"
            img1=np.array(Image.open(img_name))
            text=pytesseract.image_to_string(img1,config='--psm 6 --oem 1',lang='kor')
            img_text.append(text)
            #img_text.append('\n')
            #img_text.append(text)

            scraped_image.append({'Title': title_element.text, 'Text':text})#,'img_URL': img_url})
            #print(text)

    return scraped_image

def save_to_csv(data_list, save_path):
    if data_list:
        combined_data = pd.DataFrame(data_list)
        combined_data.to_csv(save_path, index=False)

In [ ]:
current_directory = os.getcwd()
save_path = "./data/raw/theater_ocr_before.csv"

driver = initialize_driver()
query = "http://ticket.interpark.com/TPGoodsList.asp?Ca=Dra&Sort=3"
addresslist = get_theater_links(driver, query)
scraped_image = image(driver, addresslist,"./data/posters/")
save_to_csv(scraped_image, save_path)
driver.quit()

Saved 연극 너의 목소리가 들려_part1.png
Saved 연극 너의 목소리가 들려_part2.png
Saved 연극 너의 목소리가 들려_part3.png
Saved 대학로 1위 연극 〈쉬어매드니스〉_part1.png
Saved 대학로 1위 연극 〈쉬어매드니스〉_part2.png
Saved 행오버_part1.png
Saved 행오버_part2.png
Saved 행오버_part3.png
Saved 행오버_part4.png
Saved 연극 〈바닷마을 다이어리〉_part1.png
Saved 연극 〈바닷마을 다이어리〉_part2.png
Saved 연극 한뼘사이_part1.png
Saved 연극 한뼘사이_part2.png
Saved 연극 한뼘사이_part3.png
Saved 연극 한뼘사이_part4.png
Saved 연극 한뼘사이_part5.png
Saved 연극 한뼘사이_part6.png
Saved 연극 한뼘사이_part7.png
Saved 연극 한뼘사이_part8.png
Saved 연극 한뼘사이_part9.png
Saved 연극 한뼘사이_part10.png
Saved 연극 한뼘사이_part11.png
Saved 연극 한뼘사이_part12.png
Saved 연극 한뼘사이_part13.png
Saved 연극 한뼘사이_part14.png
Saved 연극 한뼘사이_part15.png
Saved 연극〈늘근도둑이야기〉_part1.png
Saved 연극〈늘근도둑이야기〉_part2.png
Saved 뮤직드라마 〈불편한 편의점〉_part1.png
Saved 뮤직드라마 〈불편한 편의점〉_part2.png
Saved 뮤직드라마 〈불편한 편의점〉_part3.png
Saved 뮤직드라마 〈불편한 편의점〉_part4.png
Saved 뮤직드라마 〈불편한 편의점〉_part5.png
Saved 코믹감동 휴먼판타지 연극 〈2호선 세입자〉_part1.png
Saved 코믹감동 휴먼판타지 연극 〈2호선 세입자〉_part2.png
Saved 10년 연속 1위 연극〈옥탑방고양이〉- 틴틴홀_part

## 전처리 진행
-연극

In [ ]:
a=pd.read_csv("./data/raw/theater_ocr_before.csv")
a

,Title,Text
0,연극 너의 목소리가 들려,"0] 임이 79 90890 0000\n` 머크 <터의[목소리[""들려/ 대\n』 ..."
1,연극 너의 목소리가 들려,"@\n> `\n2\n1 >\n“> ㅜ\n, ..."
2,연극 너의 목소리가 들려,ㅇ\n1077\n2100 430 6130\n박선영 박선영 피설아 : 아 인 아 인 ...
3,대학로 1위 연극 〈쉬어매드니스〉,”-- -ㅋ- =\n콘텐츠박스 ...
4,대학로 1위 연극 〈쉬어매드니스〉,오이타 내더연어\n그 오소시 거으가\n.0010066 10월\n...
...,...,...
255,서교동에서 죽다 - 서울 꿈빛극장,"가 ~ ~ …~\n- “""% ㅋ\n주 ~ ..."
256,섬,@ ' ...
257,섹시코미디연극 나의PS파트너 (청주),때 다 이 [ 1\n내가 그 날 열마Ｌ\n~ 『\...
258,쇼팔다 미친 유령 극단,"8 마지막 주, 10잇 번페 주 공연 없음\n2023년 9월 1일부터 10월 15일..."


In [ ]:
result_df = a.groupby("Title")["Text"].apply(lambda x: "\n".join(x)).reset_index()
result_df

,Title,Text
0,!로맨틱코미디 연극〈운빨로맨스〉! - 대구,| = [그 ...
1,(리얼타임 코믹연극) 택시안에서 - 부산,"、 、 웃음, 감동, 사랑이 시작되는 리얼타엄 코믹연극\n2019.03.15680)..."
2,(리얼타임 코믹연극) 택시안에서 - 서울,ㆍ ㆍ ㆍ 0” 로 변\...
3,(코믹연극) 달동네-부산,0 ~ 20006 웅새떼스 026 토\n그 206 6: 버 ~? ...
4,10년 연속 1위 연극〈옥탑방고양이〉- 틴틴홀,가을 바람 살학살항\n뼈 |\n로맨틱\n|\n205\n10넌 연숙 ...
...,...,...
151,［연극 on stage］ 띨뿌리 - 화성,"크 의닝 아Ｌ 1\n\n트리거워닝안내 ” ㅠㅠ ~,\n\n연극 <떨뿌리..."
152,［연애하기 좋은 날 : 당근거래］ - 전주,"~ 오 100 ^ 1\n. -헤\n개 |"" 0 ..."
153,［장애인 선예매］ 러브 앤 인포메이션,...
154,［창작공감: 연출］당신에게 닿는 길,9 홍익대대학로 작연출:\...


In [ ]:
#불용어 지정

stopwords = "증정선물은 변동 예매 휘 전주덕진공원 타거 전주 전주덕진예술회관 전북 광주방국 헤 르 서얼 세븐일레븐 로잔티움파크 예름포차 오게 거 벼으해 나이일 경주지사 정살이 수정빈 출 광주광역시 서구 내방로 쌍촌동 전화 전주시 덕진구 권삼득로 가장가까운 버스역 수도시설사업소 번중리동급행 번 번 주차장 시설 내에 위치 무료주차 가능 평일 작 이민구 연출 손현규 협력연출 정보는 공연시작 한달전확인가능합니다 정지오 무대미술 뻐터퍼이이이 유주영 조명디자인 김범기 출연 김단아 유소령 새재 너 미령 외 다른 련 하 배역감독 역 김단아 보라 외 다른 배역 예지 경찰 직원 미령의 유소령 거스 레 드 들 태 아스 으로 후로 는개 으비세니은 오버 모 넷 늘적 팔객 마 는읍느느느능들 토요일 사 시유 커피스미스 제 연 우쑤푸 을 래 유니클로 제 제작주치극단 바라 소오으수 으으 미니티켓 아격 균일 원 다 가 해노 우이 기오 베 느슬는 아우 스오 되 시누 경성대 부경대역 빠 부산 남구 용소로 번길 부산 해바라기 소극장 대연동 대명빌딩 채 뿌 부산 오후 윤영식 오현석 허진 오후 은영식 오현석 허 별 오후 윤영식 오현석 허진 오호 윤이 오현석 허진 오후 윤영식 곽희규 인인 오 윤 식 곽희규 안애린 수 오후 곽회규 김재 옥 오후 오현석 김광재 허진 에 오후 오현석 곽회규 허진 오후 윤영식 김공저 허진 도 오후 윤 식 김재 허진 오후 곽회규 김광재 허진 월일 오소 곽회규 김광재 허진 오후 오헨석 김광재 허진 오호 오리석 김광재 허진 수 오후 윤영식 곽회규 인애린 적 목 오후 윤영식 곽희규 허진 쿠께 오후 곽회규 김광재 허진 오후 윤영식 곽회규 허진 오후 윤영식 곽회규 힌 오후 오런석 김광재 허진 오호 오린석 김광재 허진 수 오후 오현석 곽회규 허진 목 오후 윤영식 오현석 허진 오후 윤영식 오현석 안애린 오후 곽회규 김광재 인인 오후 곽회규 김재 인애린 오후 오헨석 김광재 오후 오렌석 김광재 허진 수 오후 윤 식 곽회규 허진 목 오후 윤영식 오현석 오후 곽희규 김광재 허진 오후 곽회규 김재 인애린 오후 곽회규 김광재 안애린 오후 윤영식 오현석 허진 오호 윤영식 오현석 허전 본 캐스팅 기획사 사정에 따라 예고없이 변경 될 수 있습니다 나 울째 회 나아 주 해 주 오제 에 널 더 저 후 구조 스즈 나 거우 로 보 에 퀴 키크 거즈 하어거어엇격빈대 겨 기 그 금 일 부산해바라기 아트플러스씨어터 화 분 토 시 및 공휴일 낮 공연 단체관람 문의 롯데시네마 도이 에이바헤어 아트플러스씨어터 화담 할리스 서브웨이 주소 대구 증구 동성로 길 지하소극장 국장예 별도의 주자장미 마련되머 있지 않습니다 주변 교톨미 혼잡하오니 인근 유로 주자장을 이용하거나 가급적 대증교통을 이용 바랍니다 낮 공연 및 단체문의 그 으삐 연극 월 배우 스케줄 꼴 미누 스케흘 까 타키 문재웅 조은진 이주영 신서진 노오으으으오오오 오으으오오고 김로언 오하성 김진화 오추 김인 오하성 전청일 오하성 서예선 전청일 오헝 서이선 한상웅 오하성 김진화 조소 한상웅 오하성 김진화 김로언 김나온 이차연 전청일 오하성 김진화 써 한상웅 오하성 이채연 전청일 한재우 김진화 김로언 한재우 김진화 김로언 오하성 서에선 전청일 김나온 이채연 오츄 전청일 김니온 이차연 전청일 오하성 서에선 오으 전청일 오하성 서예선 김로언 한재우 이차연 한상웅 한재우 이채연 내 한상웅 김나온 서언 을 김로언 김나온 한상웅 오하성 한상웅 오하성 김진화 한상웅 오하성 세션 전청일 오하성 김진화 오총 전청일 오라성 김로언 김나온 이채연 전청일 오하성 이채연 한상웅 한재우 한상웅 김나온 이채면 전청일 오하성 서에선 전청일 오하성 렬 김로언 한재우 김로언 김나온 오노 김로언 김나온 김로언 김나온 서에선 앨 김로언 김나온 이차연 전청일 한재우 서 선 한상웅 김나온 서에선 김로언 한재우 이채면 혈 달동데로 놈러요세 쓰이 나래 쁜 대 의 내로 낼 우소 사시 여 러 개기 쏘 소스 콜리 패 초콜릿 팩토 경성대학교 정문 자 아트박스 조흐엇 마드 능얼오영 센쥬리시티 오피스텔 교보문고 루주소 부산광역시 수영로 신암빌딩 초콜릿팩토리 로대중교통 지하철 호선 번출구 앞 버스 경성대학교 입구 하차 루건몰 유료 주차장이 있으니 참고 부탁드립니다 공연문의 소 한 드재슨구엔체이 계 당도네 고속 니 요일 시간 경민 혜자 병구 정용 정음 바벨 오루 괴획규 양재 정벌진 양차은 곽화규 양준재 정범진 양채은 조영준 양준재 정범진 양채은 인 오로 조영준 양준재 정범진 양채은 뼈 오로 조준 점유나 양준재 정봄진 임유림 이식 조영준 정유나 양준재 정범진 임유림 오루 조준 정아 연준 정유나 잉준재 정범진 임유림 조영준 양채은 정창운 임유림 버 글 조영준 인아린 양준재 정범진 잉짜은 인 오두 인마린 정창운 양자은 으 정창운 양채은 오조 조영주 정유나 오련석 양준재 임유림 시께 조영준 정유나 양준재 임유림 버 인 조영주 양준재 정범진 양자은 조영준 양준재 정범진 양차은 정유나 양준재 정칭운 잉체은 양처은 양주재 정범진 임유림 티 조영준 정유나 정범진 임유림 인 인 오두 조준 정아 정범진 임유림 조영준 정유나 정범진 임유림 방 모리아 오루 조영준 안매린 곽하구 양준재 양차은 네티 조영준 곽화규 양준재 양차은 디 조영준 정유나 잉준재 정범진 양차은 오루 조준 점유나 양주재 정범진 양은 따리 조영주 정유 곽희 정칭운 양처은 오일거일 조영준 체은 석 양재 임뮤림 조영준 양채은 양준재 임유림 은루 곽화구 양준재 정창운 임유림 따이 양준재 정창운 임유림 조영준 잉차은 양준재 정범진 임유림 조잉주 정유나 양재 정범진 잉짜은 조영준 정창운 양채은 정유나 양준재 정범진 임유림 인 정유나 양준재 정범진 임유림 오일자인 조준 아 시온 노가이 구넌 보고 쑤 가드 또 배게 루이리 오스 세이 수시 뜨태아스 매 넌 그가 가스 니나 오스 따오 따기 후오어 사우 서거 스어 이오 자라 규개 고아즈자 다즈 네 녀 여일 타 넬기비기 하던가 를 빼 니다 오너 기덜벨리 노기 아는 니이 으시 스 터는 세 배스 은 오고 혹 미 곱슬 챙 뽑 아빠 후사 초 이바 뱃 떼매제 에메 이에 원터파크 맛 우 김동진 함원태 김건아 이선준 깨 때 점지호 임채민 활민환 양시환 심채마 점한슬 김동진 이선준 점승지 문지원 정끼호 점승지 임채민 문지원 점승지 임세민 점피호 문지원 점송지 점피호 문피원 장진후 최지환 활민환 잠진호 양시환 최지환 장진호 왕시한 활민환 최지환 잠진후 함민환 최지환 미주하 바지민 심퍼마 미주하 점한술 박지민 미주하 점한술 심채마 박지민 이주하 심재마 박피민 환원태 김건하 김동진 함원태 미선준 김건하 환원태 이선준 김동진 김건하 황원태 김동진 김건하 노 임제민 문지원 정지호 점지 문지원 점피 점지호 문지원 임체민 점승지 양시환 잠진호 활민환 활민환 최지환 최지환 환민환 최지환 양시환 잠진호 점한술 박지민 심제마 미주하 박지민 미주하 심재마 박지민 점한술 이주하 미선준 김건하 김동진 함원태 긴건하 환원태 김동진 김건하 미선준 함원태 분지원 임제민 점지 문지원 정스지 임체민 정지 문지원 점지호 최지환 양시란 장진호 최지환 장진호 앙시환 잠진호 최지환 활민환 바지만 점한슬 미주하 박지민 미주하 정한술 이주하 박지민 심체마 길런하 미썬준 한윈태 김견하 한원태 선준 한원태 김건하 김동진 새 임제민 문지원 점지 점지호 점지호 임채민 문지원 점승지 점지호 임세민 양시환 최지탄 장진호 잠진호 활민환 양시환 최지환 잠진호 활민환 양한 점한솔 바피민 미주하 심체마 심채마 점한술 바피민 이주하 심체마 점한술 준 길건하 환윈태 김동진 진 미선준 김건하 한원터 김동진 미손준 점지호 문지원 활민환 최지환 심채마 박지민 김동진 김건하 스케즐은 제작사 사점미 의해 사전 공지 없이 변경될 로채사아 어렇고 인가르 조수 예성 구세 치 억 당 데 이애여서 브으 칭출지만 아끼오 아거에 노오 액느 은새소 철이 았 술사 애그 아서버 어마 이기 게서 니사 더스 스때 딘 봄 아버소아사티 오네 아이 시주 넌 뱅아수우 도아 바 트여 입 개그 소서노 우애 바소 으무수 가가  정유나 정칭운 양자은 조영준 정유나 곽희 정창운 양채은 캐스탐배우외키퇴사사정 따라며고었이변경밀수 바바 벌초 김로언 한재우 웅새떼스 버 삐 어느 시아 원떼 버 시나 우소 이채연 전청일 쓰 오하성 서예선 배 접 협 한상웅 오하성 김좌 오휴 싱옹 오성 김진화 김로언 작연출 한민규 우범진 김범진 만드는 만드는 안무 양은숙 김시명 사람들 유다미 이다혜 조 김광훈 이상은 의상 이윤진 이수연 음악 유수진 전정욱 음랄 노르 아아 안세운 조승연 상 정병목 장주희 조연출 이준성 서울시전문예술단체 극단 혈우극단 대표 주요작품 오하성 서예선 아어 문재용 문재뭉 고 이 조은진 아르 둘재웅 개 오이 공연없음 소노오 느무 그려 브 는 절선우 "


# 텍스트 전처리 함수 정의
# 영어,기호제거

def preprocess_text(text):

    # 구두점 및 특수 문자 제거
    text = re.sub(r'[^\w\s]', '', text)

    cleaned_text = re.sub(r'[^가-힣0-9]', ' ', text)

    # 토큰화 (단어 단위로 분리)
    tokens = nltk.word_tokenize(cleaned_text)


    # 불용어 제거
    stop_words = set(stopwords.split(' '))
    filtered_tokens = [word for word in tokens if word not in stop_words]

    return ' '.join(filtered_tokens)



# 텍스트 전처리 함수 정의
#숫자도 제거-영어,기호제거

def preprocess_text_num(text):

    # 구두점 및 특수 문자 제거
    text = re.sub(r'[^\w\s]', '', text)

    cleaned_text = re.sub(r'[^가-힣]', ' ', text)

    # 토큰화 (단어 단위로 분리)
    tokens = nltk.word_tokenize(cleaned_text)

    # 불용어 제거
    stop_words = set(stopwords.split(' '))
    filtered_tokens = [word for word in tokens if word not in stop_words]

    return ' '.join(filtered_tokens)


result_df['Text_clear'] = result_df['Text'].apply(preprocess_text)
result_df['Text_clear_num'] = result_df['Text'].apply(preprocess_text_num)

In [ ]:
result_df

,Title,Text,Text_clear,Text_clear_num
0,!로맨틱코미디 연극〈운빨로맨스〉! - 대구,| = [그 ...,20230901금 20231029일 7시 30분 3시 6시 2시 5시 0106460...,운빨 로맨스 으릉 점에 살고 점에 죽는 점보늬의 오랑 남자와 하롯밤 프로젝트 목숨이...
1,(리얼타임 코믹연극) 택시안에서 - 부산,"、 、 웃음, 감동, 사랑이 시작되는 리얼타엄 코믹연극\n2019.03.15680)...",웃음 감동 사랑이 시작되는 리얼타엄 코믹연극 20190315680 0061 공연시간...,웃음 감동 사랑이 시작되는 리얼타엄 코믹연극 공연시간 투 콩감백배 다양한에피소드와 ...
2,(리얼타임 코믹연극) 택시안에서 - 서울,ㆍ ㆍ ㆍ 0” 로 변\...,0 변 웃음 감동 사랑이 시작되는 리얼타엄 코믹연극 20181012080 024 씨...,변 웃음 감동 사랑이 시작되는 리얼타엄 코믹연극 씨해바라기 공연시간 광일 뽀 계묵주...
3,(코믹연극) 달동네-부산,0 ~ 20006 웅새떼스 026 토\n그 206 6: 버 ~? ...,0 20006 026 206 6 1 220 원떼00698600600006900000...,전사 오픈런 부족 초대합니다 개스 테 그를 코 뻔 부 토사 사으소시 서사으 냈 이거...
4,10년 연속 1위 연극〈옥탑방고양이〉- 틴틴홀,가을 바람 살학살항\n뼈 |\n로맨틱\n|\n205\n10넌 연숙 ...,가을 바람 살학살항 로맨틱 205 10넌 연숙 위인데 5 0 0 44 23 0000...,가을 바람 살학살항 로맨틱 연숙 위인데 단하나의첨춘로맨스 배트 토니 잘 랄라 목탑방...
...,...,...,...,...
151,［연극 on stage］ 띨뿌리 - 화성,"크 의닝 아Ｌ 1\n\n트리거워닝안내 ” ㅠㅠ ~,\n\n연극 <떨뿌리...",크 의닝 1 트리거워닝안내 떨뿌리는 극의 흐름상 일부장면에서 62 폭격음과 같은 큰...,크 의닝 트리거워닝안내 떨뿌리는 극의 흐름상 일부장면에서 폭격음과 같은 큰 소리 폭...
152,［연애하기 좋은 날 : 당근거래］ - 전주,"~ 오 100 ^ 1\n. -헤\n개 |"" 0 ...",100 1 0 66 평점 95점0 1 1즌1 영한 제작 이져 1 재관람 관객 만명 ...,평점 점 즌 영한 제작 이져 재관람 관객 만명 돌파 범 마가 청 걸 교 만 관걱이선...
153,［장애인 선예매］ 러브 앤 인포메이션,...,러브 앤 인포메이션 내키 1006 800 10600002100 울어 뜨으 포구 응극...,러브 앤 인포메이션 내키 울어 뜨으 포구 응극 용으 극 겔 려 진해정 씨 공연예술 ...
154,［창작공감: 연출］당신에게 닿는 길,9 홍익대대학로 작연출:\...,9 홍익대대학로 21 아트센터소극장 6 너6기 라3 공연시간 티켓 가격 19시 30...,홍익대대학로 아트센터소극장 라 공연시간 티켓 가격 전석 만 천원 토일 시월 할인관련...


In [ ]:
#불용어 처리 잘 되었는지 확인
result_df['Text_clear_num'][0]

'운빨 로맨스 으릉 점에 살고 점에 죽는 점보늬의 오랑 남자와 하롯밤 프로젝트 목숨이 걸린 중요한 미션 과연 보늬의 문명은 뿔 으어 을주을 간의 호랑이미 낭자와 년 아오 화롯을 보니지잖으면 바나 서성의모든볼운은 으리 본인으로부터 시작된다고 믿는 훈명은 스스 컨누 개척하는 것 점보늬가 사는 건물의 새건물주이자 년생 숫총각 호랑이띠 호 저발로 오왁한브 첫 꿀애 반한다는 전믿습니 체절 두 야 믹 부시 바브 덜 드라마 뮤지컬 등에서 자신들만의 통통 튀는 매력으로 뉴스 사랑받고 있는 실력파 배우들의 싱크로율 살아있는 연기 차여 뉴스컬처 불필요하게 늘어지는 장면 빠르게 진행되는 흐름은 고구마를 먹은 듯한 답답함을 느낄 여지를 주지 않는다 문화뉴스 동명원작웹툰 운빨로맨스를 연극의 특성을 담아 각색하며 많은 준비를 통해 운빨로맨스만의 아이덴티티를 구축 비 설렘 폭발 여기가 로코맛집 갔 간의 특별한 로맨틱 코미디 진실의 광대 광대야 진정해 게미맛집 벼 대학로 데이트 코스 위 운빨로맨스가 전하고 싶은 메시지 과 포기하지 말아요 제택후 이건 운이 아니라 능력입니다 부니 기는 산국 점보늬 제가 꼭 호랑이띠 남자와 하롯밤을 보내야 하거든요 한량하 전 카페 카사블랑카의 호 노월희 의멀티 저는 노월희 제대리님 오피스 와이퍼에요 티이 곤이 설 맨 호 여대 켜 어 궁술 오시는 운빨로맨스'

In [ ]:
#csv파일로 저장
result_df.to_csv('./data/processed/theater_ocr_processed.csv')

In [ ]:
a=pd.read_csv("./data/processed/theater_ocr_processed.csv")
a

In [ ]:
a.to_excel('./data/processed/theater_ocr_processed.xlsx')